# PTB-XL InceptionTime + BiGRU training setup

Notebook này chuẩn bị Dataset/DataLoader, kiểm tra split leakage và khởi tạo mô hình multimodal: ECG `(B, 12, 5000)` + clinical `[age_z, sex]`, đầu ra 5 superclass. Backbone là BiGRU hai chiều (96 hidden units mỗi hướng, 2 lớp).


In [ ]:
from pathlib import Path
import ast
import sys

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

current_dir = Path.cwd().resolve()
project_root = next(
    (path for path in (current_dir, *current_dir.parents)
     if (path / 'data' / 'processed' / 'ptbxl_recordwise_leadwise_zscore_500hz.npy').is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError('Open this notebook from the project or a subfolder of it.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
data_dir = project_root / 'data'
processed_dir = data_dir / 'processed'
signal_path = processed_dir / 'ptbxl_recordwise_leadwise_zscore_500hz.npy'
signal_index_path = processed_dir / 'ptbxl_recordwise_leadwise_zscore_500hz_index.csv'
clinical_path = processed_dir / 'ptbxl_clinical_features_age_zscore_sex_binary.csv'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Project:', project_root)
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

## Rebuild leakage-safe splits and quarantine

Dùng cùng quy tắc quarantine form/rhythm-only của notebook tiền xử lý. Chỉ fold 1–8 cho train, fold 9 validation, fold 10 test; loại toàn bộ patient thuộc nhóm quarantine khỏi cả ba tập.

In [ ]:
from src.data import PTBXLMultiModalDataset, assert_patient_disjoint
from src.models import InceptionBiGRUClassifier
from src.engine import run_epoch
from src.data import SUPERCLASSES

metadata = pd.read_csv(data_dir / 'ptbxl_database.csv', encoding='latin-1')
scp_statements = pd.read_csv(data_dir / 'scp_statements.csv', index_col=0)
signal_index = pd.read_csv(signal_index_path)
clinical_features = pd.read_csv(clinical_path)

diagnostic = scp_statements.loc[scp_statements['diagnostic'].eq(1.0)]
superclass_lookup = {
    str(code): str(row['diagnostic_class'])
    for code, row in diagnostic.iterrows()
    if pd.notna(row.get('diagnostic_class')) and str(row['diagnostic_class']) in SUPERCLASSES
}
form_rhythm_codes = {
    str(code) for code, row in scp_statements.iterrows()
    if row.get('form') == 1.0 or row.get('rhythm') == 1.0
}

def parse_scp_codes(value):
    if pd.isna(value):
        return {}
    try:
        parsed = ast.literal_eval(value)
    except (TypeError, ValueError, SyntaxError):
        return {}
    return parsed if isinstance(parsed, dict) else {}

def is_form_rhythm_only_unmapped(value):
    codes = set(parse_scp_codes(value))
    return bool(codes) and codes.issubset(form_rhythm_codes) and codes.isdisjoint(superclass_lookup)

review_patient_ids = set(metadata.loc[
    metadata['scp_codes'].map(is_form_rhythm_only_unmapped), 'patient_id'
])
records = signal_index.merge(
    clinical_features, on='ecg_id', how='inner', validate='one_to_one', suffixes=('', '_clinical')
)
if len(records) != len(signal_index) or len(records) != len(clinical_features):
    raise ValueError('Signal index and clinical features do not have a one-to-one ECG ID match.')
metadata_patient_ids = metadata.set_index('ecg_id')['patient_id'].reindex(records['ecg_id'])
if metadata_patient_ids.isna().any() or not np.array_equal(
    metadata_patient_ids.to_numpy(), records['patient_id'].to_numpy()
):
    raise ValueError('Patient IDs in the signal index do not match source metadata.')

eligible = records.loc[~records['patient_id'].isin(review_patient_ids)].copy()
train_records = eligible.loc[eligible['strat_fold'].isin(range(1, 9))].copy()
validation_records = eligible.loc[eligible['strat_fold'].eq(9)].copy()
test_records = eligible.loc[eligible['strat_fold'].eq(10)].copy()
assert_patient_disjoint(train_records, validation_records, test_records)
print('Quarantined patients:', len(review_patient_ids))
print('Train / validation / test records:', len(train_records), len(validation_records), len(test_records))
print('Patient overlap audit: passed')

In [ ]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

train_dataset = PTBXLMultiModalDataset(
    train_records, signal_path, signal_index_path, clinical_path, expected_folds=set(range(1, 9))
)
validation_dataset = PTBXLMultiModalDataset(
    validation_records, signal_path, signal_index_path, clinical_path, expected_folds={9}
)
test_dataset = PTBXLMultiModalDataset(
    test_records, signal_path, signal_index_path, clinical_path, expected_folds={10}
)

# RTX 3050 4 GB: begin at batch_size=1; increase only after checking peak VRAM.
batch_size = 64
loader_options = {
    'batch_size': batch_size,
    'num_workers': 0,
    'pin_memory': device.type == 'cuda',
}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_options)
validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_options)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_options)

batch = next(iter(train_loader))
print('ECG batch:', tuple(batch['ecg'].shape), batch['ecg'].dtype)
print('Clinical batch:', tuple(batch['clinical'].shape), batch['clinical'][0].tolist())
print('Target batch:', tuple(batch['target'].shape), SUPERCLASSES)
print('ECG IDs:', batch['ecg_id'][:5].tolist())

In [ ]:
# On CPU, use a smaller model only to verify the forward path. Full training requires CUDA.
if device.type == 'cuda':
    model = InceptionBiGRUClassifier().to(device)
else:
    model = InceptionBiGRUClassifier(branch_channels=8, gru_layers=1).to(device)
    print('CPU smoke-test configuration: branch_channels=8, GRU layers=1; do not start full training.')

parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f'Trainable parameters: {parameter_count:,}')

model.eval()
with torch.inference_mode():
    logits = model(batch['ecg'].to(device), batch['clinical'].to(device))
    loss = torch.nn.BCEWithLogitsLoss()(logits, batch['target'].to(device))
print('Logits:', tuple(logits.shape), '| smoke-test BCE loss:', float(loss))
assert logits.shape == (batch_size, len(SUPERCLASSES))
assert torch.isfinite(logits).all()

## Train and select the best checkpoint

Each epoch reports validation macro AUPRC; the checkpoint with the highest validation macro AUPRC is retained. After training, AUROC and AUPRC are reported for each superclass and as macro averages. No threshold is optimized. A 0.5 threshold is the default only for converting probabilities to binary labels; AUROC/AUPRC are computed from probability scores and do not depend on that threshold. The test set is evaluated once in the next cell using the selected checkpoint.

In [ ]:
if device.type != 'cuda':
    raise RuntimeError('Full training is disabled on CPU in this notebook. Install CUDA-enabled PyTorch and confirm torch.cuda.is_available() is True.')

epochs = 14
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
checkpoint_dir = project_root / 'models' / 'checkpoints'
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / 'inception_bigru_best.pt'
best_validation_auprc = float('-inf')
best_epoch = None

for epoch in range(1, epochs + 1):
    run_epoch(model, train_loader, device, optimizer=optimizer, amp=True)
    validation_metrics = run_epoch(model, validation_loader, device, amp=True)
    score = validation_metrics['macro_auprc']
    print(f'Epoch {epoch:02d}/{epochs} | val macro AUPRC: {score:.6f}')
    if score is not None and score > best_validation_auprc:
        best_validation_auprc = score
        best_epoch = epoch
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'validation_macro_auprc': score,
            'clinical_feature_order': ['age_z', 'sex'],
            'labels': list(SUPERCLASSES),
        }, checkpoint_path)

if best_epoch is None:
    raise RuntimeError('No checkpoint was selected; validation macro AUPRC was unavailable.')

best_checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
model.load_state_dict(best_checkpoint['model_state_dict'])
best_validation_metrics = run_epoch(model, validation_loader, device, amp=True)
per_label_table = pd.DataFrame.from_dict(best_validation_metrics['per_label'], orient='index')
per_label_table.index.name = 'Superclass'
print(f'\nBest checkpoint: epoch {best_checkpoint["epoch"]} | validation macro AUPRC: {best_checkpoint["validation_macro_auprc"]:.6f}')
print('Validation AUROC/AUPRC by superclass:')
print(per_label_table.to_string(float_format=lambda value: f'{value:.4f}'))
print(f'Macro AUROC: {best_validation_metrics["macro_auroc"]:.4f}')
print(f'Macro AUPRC: {best_validation_metrics["macro_auprc"]:.4f}')

## Final test-set evaluation

Run once after training. The selected validation checkpoint is evaluated on test and reports AUROC, AUPRC, precision, recall, F1, and accuracy per superclass and as macro averages. Threshold-dependent metrics use the fixed default threshold 0.5; it is not optimized on validation or test. Subset accuracy is also reported as the fraction of records for which all five labels match.

In [ ]:
from src.engine import run_epoch

decision_threshold = 0.5
checkpoint_path = project_root / 'models' / 'checkpoints' / 'inception_bigru_best.pt'
if not checkpoint_path.is_file():
    raise FileNotFoundError(f'Best validation checkpoint not found: {checkpoint_path}. Run the training cell first.')

test_checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
if test_checkpoint.get('labels') != list(SUPERCLASSES):
    raise ValueError('Checkpoint label order does not match the current dataset labels.')
if test_checkpoint.get('clinical_feature_order') != ['age_z', 'sex']:
    raise ValueError('Checkpoint clinical feature order does not match the current model inputs.')
model.load_state_dict(test_checkpoint['model_state_dict'])
test_metrics = run_epoch(
    model, test_loader, device, amp=True,
    detailed_metrics=True, threshold=decision_threshold,
)
test_table = pd.DataFrame.from_dict(test_metrics['per_label'], orient='index')
test_table.index.name = 'Superclass'
print(f'Best checkpoint from epoch {test_checkpoint["epoch"]} | validation macro AUPRC: {test_checkpoint["validation_macro_auprc"]:.4f}')
print(f'Test metrics by superclass (fixed threshold = {decision_threshold:.1f} for threshold-dependent metrics):')
print(test_table.to_string(float_format=lambda value: f'{value:.4f}'))
print('Test macro metrics:')
for metric in ('macro_auroc', 'macro_auprc', 'macro_precision', 'macro_recall', 'macro_f1', 'macro_accuracy', 'subset_accuracy'):
    print(f'  {metric}: {test_metrics[metric]:.4f}')